# Phase 1 — Lesson 10: Dimensionality Reduction
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 10: Dimensionality Reduction
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 01, 02, 03 (Linear Algebra), 06 (Probability & Distributions)



# What You'll Learn

---

## Learning Objectives

- Implement PCA from scratch: center data, compute the covariance matrix, eigendecompose, and project
- Use explained variance ratio and the elbow method to choose the number of principal components
- Compare PCA, t-SNE, and UMAP for visualizing MNIST digits in 2D and explain their tradeoffs
- Apply kernel PCA with an RBF kernel to separate nonlinear data structures that standard PCA cannot handle


# The Problem

---

You have a dataset with 784 features per sample. Maybe it is pixel values of handwritten digits. Maybe it is gene expression levels. Maybe it is user behavior signals. You cannot visualize 784 dimensions. You cannot plot them. You cannot even think about them.

But most of those 784 features are redundant. The actual information lives on a much smaller surface. A handwritten "7" does not need 784 independent numbers to describe it. It needs a few: the angle of the stroke, the length of the crossbar, how much it leans. The rest is noise.

**Dimensionality reduction finds that smaller surface.** It takes your 784-dimensional data and compresses it to 2, 10, or 50 dimensions while keeping the structure that matters.


# The Curse of Dimensionality

---

High-dimensional spaces are unintuitive. Three things break as dimensions grow:

**1. Distance becomes meaningless.** In high dimensions, the distance between any two random points converges to the same value. If every point is roughly the same distance from every other point, nearest-neighbor search stops working.

```
Dimension    Avg distance ratio (max/min between random points)
2            ~5.0
10           ~1.8
100          ~1.2
1000         ~1.02
```

**2. Volume concentrates in corners.** A unit hypercube in d dimensions has 2^d corners. In 100 dimensions, nearly all the volume is in the corners, far from the center.

**3. You need exponentially more data.** Going from 2D to 20D means you need 10^18 times more data to maintain the same density. You never have enough.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)# Show distance concentration: max/min distance ratio vs dimensionfrom scipy.spatial.distance import pdistdims = [2, 10, 50, 100, 200, 500]ratios = []for d in dims:    pts = np.random.randn(500, d)    dsts = pdist(pts)    ratios.append(dsts.max() / dsts.min())plt.figure(figsize=(7, 4))plt.plot(dims, ratios, 'o-', color='#e74c3c', lw=2, markersize=6)plt.axhline(1.0, color='#27ae60', linestyle='--', alpha=0.7, label='ratio → 1')plt.xlabel('Dimensions'); plt.ylabel('max / min distance ratio')plt.title('Distance Concentration in High Dimensions', fontsize=12, fontweight='bold')plt.grid(True, alpha=0.3); plt.legend()plt.tight_layout(); plt.show()print(f"2D ratio: {ratios[0]:.2f}  100D ratio: {ratios[3]:.2f}  500D ratio: {ratios[5]:.2f}")

# PCA: Find the Directions That Matter

---

Principal Component Analysis (PCA) finds the axes along which your data varies the most. It rotates your coordinate system so the first axis captures the most variance, the second captures the next most, and so on.

The algorithm:
```
1. Center the data        (subtract the mean from each feature)
2. Compute covariance     (how features move together)
3. Eigendecomposition     (find the principal directions)
4. Sort by eigenvalue     (biggest variance first)
5. Project               (keep top k eigenvectors, drop the rest)
```

Why eigendecomposition? The covariance matrix is symmetric and positive semi-definite. Its eigenvectors are orthogonal directions in feature space. The eigenvalues tell you how much variance each direction captures. The eigenvector with the largest eigenvalue points along the direction of maximum variance.

```
Original data (2D)     ---PCA rotation--->   After PCA
Data spread in both    PC1 captures the      PC1 captures the elongated spread
x and y directions     elongated spread      PC2 captures the narrow spread
                                                     Drop PC2 → lose little info
```


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)n = 300# Correlated 2D data (elongated along a diagonal)t = np.random.randn(n) * 2x = t + np.random.randn(n) * 0.3y = 0.5 * t + np.random.randn(n) * 0.3X = np.column_stack([x, y])# PCA from scratchXc = X - X.mean(axis=0)cov = np.cov(Xc, rowvar=False)vals, vecs = np.linalg.eigh(cov)order = np.argsort(vals)[::-1]vals, vecs = vals[order], vecs[:, order]PC1 = vecs[:, 0]X_proj = Xc @ PC1fig, ax = plt.subplots(figsize=(7, 7))ax.scatter(Xc[:, 0], Xc[:, 1], alpha=0.5, color='#2c3e50', label='data')for i, (v, val) in enumerate(zip(vecs.T, vals)):    ax.quiver(0, 0, v[0]*2.5*np.sqrt(val), v[1]*2.5*np.sqrt(val),              color='#e74c3c' if i == 0 else '#27ae60', width=0.012, head_width=0.18,              label=f'PC{i+1} (var={val:.2f})')ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)ax.set_aspect('equal'); ax.set_title('PCA: Principal Components', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()print(f"Variance retained by PC1: {vals[0]/vals.sum():.4f}")print(f"Variance retained by PC2: {vals[1]/vals.sum():.4f}")

# Explained Variance Ratio & Elbow Method

---

Each principal component captures a fraction of the total variance:

```
Component    Eigenvalue    Explained ratio    Cumulative
PC1          4.73          0.473              0.473
PC2          2.51          0.251              0.724
PC3          1.12          0.112              0.836
PC4          0.89          0.089              0.925
```

When cumulative explained variance reaches 0.95, those components capture 95% of the information. Everything after is mostly noise.


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Simulated eigenvalues for 15 componentseigenvalues = np.array([5.2, 3.1, 2.0, 1.3, 0.9, 0.6, 0.4, 0.3, 0.2, 0.15, 0.1, 0.08, 0.05, 0.03, 0.02])ratios = eigenvalues / eigenvalues.sum()cumulative = np.cumsum(ratios)fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].bar(range(1, len(eigenvalues)+1), ratios, color='#2c3e50', alpha=0.7, label='Individual')axes[0].plot(range(1, len(eigenvalues)+1), cumulative, 'ro-', label='Cumulative')axes[0].axhline(0.95, color='#e74c3c', linestyle='--', label='95% threshold')axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained variance ratio')axes[0].set_title('Explained Variance per Component', fontsize=12, fontweight='bold')axes[0].legend(); axes[0].grid(True, alpha=0.3)axes[1].plot(range(1, len(eigenvalues)+1), eigenvalues, 'o-', color='#e74c3c', lw=2, markersize=6)axes[1].set_xlabel('Principal Component'); axes[1].set_ylabel('Eigenvalue')axes[1].set_title('Scree Plot (Elbow Method)', fontsize=12, fontweight='bold')axes[1].grid(True, alpha=0.3)plt.tight_layout(); plt.show()k_95 = int(np.argmax(cumulative >= 0.95)) + 1print(f"Components for 95% variance: {k_95} (cumulative = {cumulative[k_95-1]:.4f})")

# Choosing the Number of Components

---

Three strategies:
1. **Threshold.** Keep enough components to explain 90–95% of the variance.
2. **Elbow method.** Plot explained variance per component. Look for a sharp drop-off.
3. **Downstream performance.** Use PCA as preprocessing. Sweep k and measure your model's accuracy. The best k is wherever accuracy plateaus.


# t-SNE: Preserve Neighborhoods

---

t-Distributed Stochastic Neighbor Embedding (t-SNE) is designed for visualization. It maps high-dimensional data to 2D (or 3D) while preserving which points are near each other.

The intuition: in the original space, compute a probability distribution over pairs of points based on their distances. Near points get high probability. Far points get low probability. Then find a 2D arrangement where the same probability distribution holds.

Key properties of t-SNE:
- **Non-linear.** It can unfold complex manifolds that PCA cannot.
- **Stochastic.** Different runs produce different layouts.
- **Perplexity** parameter controls how many neighbors to consider (typical range: 5–50).
- **Distances between clusters in the output are not meaningful.** Only the clusters themselves are.
- **Slow** on large datasets. O(n²) by default.


# UMAP: Faster, Better Global Structure

---

Uniform Manifold Approximation and Projection (UMAP) works similarly to t-SNE but with two advantages:
- **Faster.** Uses approximate nearest-neighbor graphs instead of computing all pairwise distances.
- **Better global structure.** The relative positions of clusters in the output tend to be more meaningful than in t-SNE.


# When to Use Which

---

| Method | Use case | Preserves | Speed |
|--------|----------|-----------|-------|
| PCA | Preprocessing before training | Global variance | Fast (exact) |
| PCA | Quick exploratory visualization | Linear structure | Fast |
| t-SNE | Publication-quality 2D plots | Local neighborhoods | Slow (< 10k samples ideal) |
| UMAP | 2D visualization at scale | Local + some global structure | Medium |

**Rule of thumb:** use PCA for preprocessing and data compression. Use t-SNE or UMAP when you need to visualize structure in 2D.


# Kernel PCA: Nonlinear Manifolds

---

Standard PCA finds linear subspaces. But what if the data lies on a nonlinear manifold? A circle in 2D cannot be separated by any line. Standard PCA will not help.

**Kernel PCA** applies PCA in a high-dimensional feature space induced by a kernel function, without explicitly computing coordinates in that space — the kernel trick (same idea behind SVMs).

The algorithm:
1. Compute the kernel matrix K where Kᵢⱼ = k(xᵢ, xⱼ)
2. Center the kernel matrix in feature space
3. Eigendecompose the centered kernel matrix
4. The top eigenvectors (scaled by 1/√eigenvalue) are the projections

| Kernel | Formula | Good for |
|--------|---------|----------|
| RBF (Gaussian) | exp(−γ·‖x−y‖²) | Most nonlinear data, smooth manifolds |
| Polynomial | (x·y + c)ᵈ | Polynomial relationships |
| Sigmoid | tanh(α·x·y + c) | Neural network-like mappings |


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)# Concentric circles: standard PCA fails, kernel PCA succeedsn = 150r1 = np.random.randn(n) * 0.2 + 1.0theta1 = np.random.rand(n) * 2 * np.piinner = np.column_stack([r1 * np.cos(theta1), r1 * np.sin(theta1)])r2 = np.random.randn(n) * 0.2 + 3.0theta2 = np.random.rand(n) * 2 * np.piouter = np.column_stack([r2 * np.cos(theta2), r2 * np.sin(theta2)])X = np.vstack([inner, outer])y = np.array([0] * n + [1] * n)# Standard PCAXc = X - X.mean(axis=0)cov = np.cov(Xc, rowvar=False)vals, vecs = np.linalg.eigh(cov)order = np.argsort(vals)[::-1]W = vecs[:, order[:2]].TX_pca = Xc @ W.Tfig, axes = plt.subplots(1, 2, figsize=(10, 4))axes[0].scatter(X_pca[y==0, 0], X_pca[y==0, 1], c='#e74c3c', alpha=0.6, label='inner', s=20)axes[0].scatter(X_pca[y==1, 0], X_pca[y==1, 1], c='#27ae60', alpha=0.6, label='outer', s=20)axes[0].set_title('Standard PCA: circles overlap', fontweight='bold')axes[0].legend(); axes[0].grid(True, alpha=0.3)# Kernel PCA with RBFfrom sklearn.decomposition import KernelPCAkpca = KernelPCA(n_components=2, kernel='rbf', gamma=15, fit_inverse_transform=True)X_kpca = kpca.fit_transform(X)axes[1].scatter(X_kpca[y==0, 0], X_kpca[y==0, 1], c='#e74c3c', alpha=0.6, label='inner', s=20)axes[1].scatter(X_kpca[y==1, 0], X_kpca[y==1, 1], c='#27ae60', alpha=0.6, label='outer', s=20)axes[1].set_title('Kernel PCA (RBF): circles separated', fontweight='bold')axes[1].legend(); axes[1].grid(True, alpha=0.3)plt.suptitle('Kernel PCA Separates Nonlinear Manifolds', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Reconstruction Error

---

How good is your dimensionality reduction? You compressed 784 dimensions to 50. What did you lose?

Measure reconstruction error:
1. Project data to k dimensions: `X_reduced = X @ W_k`
2. Reconstruct: `X_hat = X_reduced @ W_k^T`
3. Compute MSE: `mean((X - X_hat)²)`

For PCA, reconstruction error has a clean relationship to explained variance:
```
Reconstruction error = sum of eigenvalues NOT included
Total variance = sum of ALL eigenvalues
Fraction lost = (sum of dropped eigenvalues) / (sum of all eigenvalues)
```

The explained variance ratio for each component:
```
explained_ratio_k = eigenvalue_k / sum(all eigenvalues)
```

Plotting cumulative explained variance against number of components gives you the "elbow" curve. Reconstruction error is also useful for anomaly detection: samples with high reconstruction error are outliers.


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# PCA class from scratchclass PCA:    def __init__(self, n_components):        self.n_components = n_components        self.components = None; self.mean = None; self.eigenvalues = None        self.explained_variance_ratio_ = None    def fit(self, X):        self.mean = np.mean(X, axis=0)        Xc = X - self.mean        cov = np.cov(Xc, rowvar=False)        vals, vecs = np.linalg.eigh(cov)        order = np.argsort(vals)[::-1]        vals, vecs = vals[order], vecs[:, order]        self.components = vecs[:, :self.n_components].T        self.eigenvalues = vals[:self.n_components]        total = np.sum(vals)        self.explained_variance_ratio_ = self.eigenvalues / total        return self    def transform(self, X):        return (X - self.mean) @ self.components.T    def fit_transform(self, X):        self.fit(X); return self.transform(X)# Test on synthetic datanp.random.seed(42)n_samples = 500t = np.random.uniform(0, 2 * np.pi, n_samples)x1 = 3 * np.cos(t) + np.random.normal(0, 0.2, n_samples)x2 = 3 * np.sin(t) + np.random.normal(0, 0.2, n_samples)x3 = 0.5 * x1 + 0.3 * x2 + np.random.normal(0, 0.1, n_samples)X_synthetic = np.column_stack([x1, x2, x3])pca = PCA(n_components=2)X_reduced = pca.fit_transform(X_synthetic)print(f"Original shape: {X_synthetic.shape}")print(f"Reduced shape:  {X_reduced.shape}")print(f"Explained variance ratios: {np.round(pca.explained_variance_ratio_, 4)}")print(f"Total variance captured: {sum(pca.explained_variance_ratio_):.4f}")# Reconstruction error vs kks = range(1, 4)recon_errors = []for k in ks:    p = PCA(n_components=k)    Xr = p.fit_transform(X_synthetic)    # reconstruct    X_hat = Xr @ p.components + p.mean    mse = np.mean((X_synthetic - X_hat) ** 2)    recon_errors.append(mse)plt.figure(figsize=(7, 4))plt.plot(ks, recon_errors, 'o-', color='#e74c3c', lw=2, markersize=6)plt.xlabel('Number of components (k)'); plt.ylabel('Reconstruction MSE')plt.title('Reconstruction Error vs Number of Components', fontsize=12, fontweight='bold')plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()for k, err in zip(ks, recon_errors):    print(f"k={k}: reconstruction MSE = {err:.6f}")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does PCA actually do?
> **Explanation:** Rotates the coordinate system so axes align with directions of maximum variance, then drops low-variance axes.

**Q2:** What does explained variance ratio tell you?
> **Explanation:** The fraction of total variance captured by one principal component. Sum top-k ratios to see how much k components preserve.

**Q3:** When does standard PCA fail but kernel PCA succeed?
> **Explanation:** When data lies on a nonlinear manifold (e.g., concentric circles) — standard PCA finds only linear structure.

**Q4:** What is the curse of dimensionality?
> **Explanation:** Distances, volumes, and data density behave counterintuitively in high dimensions; models need exponentially more data.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Modify the PCA class to support `inverse_transform`. Reconstruct MNIST digits from 10, 50, and 200 components. Print the reconstruction error (MSE) for each.

**Exercise 2:** Run t-SNE on the same MNIST subset with perplexity values of 5, 30, and 100. Describe how the output changes. Why does perplexity affect cluster tightness?

**Exercise 3:** Take a dataset with 50 features where only 5 are informative (generate with `sklearn.datasets.make_classification`). Apply PCA and check whether the explained variance curve correctly identifies that the data is effectively 5-dimensional.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Curse of dimensionality | "Too many features" | Distances, volumes, and data density behave counterintuitively as dimensions grow. Models need exponentially more data to compensate. |
| PCA | "Reduce dimensions" | Rotate your coordinate system so axes align with directions of maximum variance, then drop low-variance axes. |
| Principal component | "An important direction" | An eigenvector of the covariance matrix — the direction in feature space along which data varies most. |
| Explained variance ratio | "How much info this component has" | The fraction of total variance captured by one principal component. |
| Covariance matrix | "How features correlate" | A symmetric matrix where entry (i,j) measures how feature i and feature j move together. Diagonal = individual variances. |
| t-SNE | "That cluster plot" | A nonlinear method mapping high-D data to 2D by preserving pairwise neighborhood probabilities. Good for visualization, not preprocessing. |
| UMAP | "Faster t-SNE" | A nonlinear method based on topological data analysis. Preserves local and some global structure. Scales better than t-SNE. |
| Perplexity | "A t-SNE knob" | Controls the effective number of neighbors each point considers. Low = local structure; high = broader patterns. |
| Manifold | "The surface data lives on" | A lower-dimensional surface embedded in a higher-dimensional space (e.g., a crumpled sheet of paper is a 2D manifold in 3D). |


# Further Reading

---

- [A Tutorial on Principal Component Analysis](https://arxiv.org/abs/1404.1100) (Shlens) — clear derivation of PCA from the ground up
- [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/) (Wattenberg et al.) — interactive guide to t-SNE pitfalls and parameter choices
- [UMAP documentation](https://umap-learn.readthedocs.io/) — theory and practical guidance from the UMAP authors


# Next Steps

---

When you're ready, continue to Lesson 11: Singular Value Decomposition.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
